# Histórico de decisões: Cats and Dogs

In [1]:
import glob
import json
import re

from IPython.display import Markdown, display

def tabela(cabecalho, linhas):
    texto = "| " + " | ".join(cabecalho) + " |\n|" + "---|" * len(cabecalho) + "\n"
    texto += "\n".join("| " + " | ".join(str(v) for v in linha) + " |" for linha in linhas)
    display(Markdown(texto))

def por_seed(valores):
    return " / ".join(valores)

## 1. Configuração inicial

| Item | Valor |
|---|---|
| Seed | 18 |
| Tamanho das imagens | o da imagem de menor área |
| Pixels | de 0 a 255, como estão |
| Aumento de dados | versões fixas: 10% giradas em 15° e 10% com ruído de média 0 e desvio 2 |
| Divisão | 15% para teste e 15% do restante para validação, por grupos |
| Convoluções | 5 blocos: 3 filtros 5×5, stride 1, padding 1, ReLU, max pooling 2×2 e dropout de 20% |
| Camadas conectadas | 1024, 32 e 2 neurônios, cada uma seguida de ReLU, e depois softmax |
| Inicialização | normal com média 0 e desvio 0,1, em pesos e bias |
| Otimização | SGD, learning rate 10⁻⁴, batch de 32, 100 épocas, entropia cruzada |

## 2. Primeiros resultados na validação


-> Ensaio feito pelo Claude: São 6.000 imagens quadradas, sendo 1.000 de validação, sem aumento de dados, por 30 épocas. O conjunto de teste não foi usado. 

In [2]:
linhas = []
for texto in open("historico/ensaio_30_epocas.txt", encoding="utf8"):
    m = re.match(r"t=(\d+) seed=(\d+) .*ep30 \(([\d.]+), ([\d.]+), ([\d.]+)\)", texto)
    if m:
        linhas.append([f"{m[1]} px", m[2], m[3], m[4], m[5]])
tabela(["Tamanho", "Seed", "Perda no treino", "Acurácia na validação", "Fração de saídas todas zero"], linhas)

| Tamanho | Seed | Perda no treino | Acurácia na validação | Fração de saídas todas zero |
|---|---|---|---|---|
| 100 px | 18 | 0.6931 | 0.489 | 0.82 |
| 100 px | 19 | 0.6934 | 0.46 | 1.0 |
| 100 px | 20 | 0.6934 | 0.498 | 0.6 |
| 128 px | 18 | 0.6934 | 0.489 | 1.0 |
| 128 px | 19 | 0.6935 | 0.46 | 1.0 |
| 128 px | 20 | 0.6934 | 0.468 | 1.0 |
| 224 px | 18 | 0.6952 | 0.519 | 0.64 |
| 224 px | 19 | 0.6931 | 0.455 | 0.97 |
| 224 px | 20 | 0.6964 | 0.526 | 0.0 |

## 3. Variantes testadas sobre a rede 

-> Ensaio realizado pelo Claude, para testar combinações e finalizar com a melhor (feito no ensaio para diminuir o tempo que seria necessário para treinar todas as variantes).

In [6]:
grupos = {}
padrao = r"(\d+)px (\d+)ep \| (.+?) \| seed (\d+) \| val final ([\d.]+) melhor ([\d.]+) \| perda treino (\S+) \| (\d+)s"
for texto in open("historico/variantes.txt", encoding="utf8"):
    m = re.match(padrao, texto)
    grupos.setdefault((m[3], int(m[1]), int(m[2])), {})[int(m[4])] = m

ordem = ["início", "sem ReLU final", "pixels/255", "sem ReLU final + pixels/255", "sem ReLU final + lr 1e-2",
         "sem ReLU final + pixels/255 + lr 1e-2", "sem ReLU final + pixels/255 + lr 1e-2 + init padrão",
         "sem ReLU final + pixels/255 + lr 1e-2 + 32 filtros",
         "sem ReLU final + pixels/255 + lr 1e-2 + 32 filtros + init padrão"]
linhas = []
for nome, px, ep in sorted(grupos, key=lambda k: (ordem.index(k[0]), k[1], k[2])):
    g = grupos[nome, px, ep]
    linhas.append([nome, px, ep, por_seed([g[s][5] for s in (18, 19)]), por_seed([g[s][6] for s in (18, 19)]), por_seed([g[s][7] for s in (18, 19)])])
tabela(["Variante", "Tamanho (px)", "Épocas", "Validação final", "Melhor validação", "Perda no treino"], linhas)

| Variante | Tamanho (px) | Épocas | Validação final | Melhor validação | Perda no treino |
|---|---|---|---|---|---|
| início | 128 | 100 | 0.489 / 0.460 | 0.489 / 0.460 | 0.693 / 0.693 |
| início | 128 | 500 | 0.489 / 0.460 | 0.489 / 0.460 | 0.693 / 0.693 |
| sem ReLU final | 128 | 100 | 0.526 / 0.496 | 0.544 / 0.525 | 0.673 / 0.688 |
| pixels/255 | 128 | 100 | 0.489 / 0.460 | 0.489 / 0.540 | 0.693 / 0.693 |
| sem ReLU final + pixels/255 | 128 | 100 | 0.489 / 0.540 | 0.489 / 0.540 | 0.692 / 0.694 |
| sem ReLU final + lr 1e-2 | 128 | 100 | 0.512 / 0.462 | 0.517 / 0.543 | 0.693 / 0.606 |
| sem ReLU final + pixels/255 + lr 1e-2 | 128 | 100 | 0.586 / 0.515 | 0.633 / 0.687 | 0.587 / 0.581 |
| sem ReLU final + pixels/255 + lr 1e-2 + init padrão | 128 | 100 | 0.511 / 0.540 | 0.511 / 0.540 | 0.693 / 0.693 |
| sem ReLU final + pixels/255 + lr 1e-2 + 32 filtros | 100 | 100 | 0.674 / 0.754 | 0.675 / 0.762 | 0.523 / 0.488 |
| sem ReLU final + pixels/255 + lr 1e-2 + 32 filtros | 128 | 100 | 0.489 / 0.552 | 0.489 / 0.570 | nan / 0.686 |
| sem ReLU final + pixels/255 + lr 1e-2 + 32 filtros | 224 | 100 | 0.489 / 0.460 | 0.489 / 0.460 | nan / nan |
| sem ReLU final + pixels/255 + lr 1e-2 + 32 filtros + init padrão | 100 | 100 | 0.675 / 0.670 | 0.675 / 0.781 | 0.558 / 0.454 |
| sem ReLU final + pixels/255 + lr 1e-2 + 32 filtros + init padrão | 128 | 100 | 0.616 / 0.675 | 0.665 / 0.743 | 0.586 / 0.517 |
| sem ReLU final + pixels/255 + lr 1e-2 + 32 filtros + init padrão | 224 | 100 | 0.638 / 0.672 | 0.652 / 0.712 | 0.481 / 0.573 |

## 4. Modelo fechado
Com as decisões acima, o notebook `cats_and_dogs.ipynb` foi rodado do início ao fim. O teste foi medido uma vez, depois de fechar o modelo.

In [7]:
h = json.load(open("historico/historico_treino.json"))
val = h["acc_val_modelo_fechado"]
ep = h["modelo_fechado_melhor_epoca"]
tabela(["Época", "Acurácia na validação"], [[e, f"{val[e - 1]:.3f}"] for e in (1, 25, 50, 75, 100)])
print(f"Melhor época: {ep} | validação: {val[ep - 1]:.3f}")
print("Teste (célula 20 de cats_and_dogs.ipynb): 4066 de 4468 = 0.9100")

| Época | Acurácia na validação |
|---|---|
| 1 | 0.504 |
| 25 | 0.699 |
| 50 | 0.871 |
| 75 | 0.895 |
| 100 | 0.903 |

Melhor época: 84 | validação: 0.905
Teste (célula 20 de cats_and_dogs.ipynb): 4066 de 4468 = 0.9100
